In [55]:
import sys, os
sys.path.insert(0, os.path.abspath("./src"))

from pandas_ta import ema


ImportError: Numba needs NumPy 2.2 or less. Got NumPy 2.3.

In [1]:
import numpy, numba
print("✅ NumPy version:", numpy.__version__)
print("📂 NumPy loaded from:", numpy.__file__)
print("✅ Numba version:", numba.__version__)
print("📂 Numba loaded from:", numba.__file__)


✅ NumPy version: 2.2.2
📂 NumPy loaded from: /home/hoang-pham/Documents/bot_v2/.venv/lib/python3.12/site-packages/numpy/__init__.py
✅ Numba version: 0.61.2
📂 Numba loaded from: /home/hoang-pham/Documents/bot_v2/.venv/lib/python3.12/site-packages/numba/__init__.py


In [16]:
import sys
import os

ROOT = "/home/hoang-pham/Documents/bot_v2/src"  # folder chứa pandas_ta
sys.path.append(ROOT)

print("ROOT:", ROOT)
print("Folders:", os.listdir(ROOT))  # phải thấy pandas_ta

from pandas_ta.momentum.macd import macd
print(macd)


ROOT: /home/hoang-pham/Documents/bot_v2/src
Folders: ['pandas_ta', 'v2', 'v1']
<function macd at 0x7772a4f980e0>


In [12]:
import sys
SRC_PATH = sys.path.append("../..")  # đường dẫn tới folder chứa pandas_ta


print("SRC_PATH:", SRC_PATH)


SRC_PATH: None


In [ ]:
from pandas_ta.momentum.macd import macd
import pandas as pd

# ==========================
# 1️⃣ Settings
# ==========================
DATA_PATH = '../data/EURUSD_5M.csv'
START_DATE = '2002-10-21'
END_DATE = '2005-11-01'

INITIAL_CAPITAL = 1000
RISK_PER_TRADE = 0.01  # % vốn mỗi trade
MAX_LOT = 2.0
PIP_VALUE = 10

STOP_LOSS_PIPS = 20
TAKE_PROFIT_PIPS = 40  # 2:1 R:R

# ==========================
# 2️⃣ Load dữ liệu
# ==========================
df = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
df = df[(df['datetime'] >= START_DATE) & (df['datetime'] <= END_DATE)]

# ==========================
# 3️⃣ Tính MACD & tín hiệu
# ==========================
close = df['close']
macd_df = macd(close)
df = pd.concat([df, macd_df], axis=1)

# Tín hiệu: 1=BUY, -1=SELL
df['signal'] = 0
df.loc[df['MACD_12_26_9'] > df['MACDs_12_26_9'], 'signal'] = 1
df.loc[df['MACD_12_26_9'] < df['MACDs_12_26_9'], 'signal'] = -1

# ==========================
# 4️⃣ Backtest đơn giản
# ==========================
capital = INITIAL_CAPITAL
trade_history = []

for i in range(len(df)-1):  # dùng candle kế tiếp để check TP/SL
    row = df.iloc[i]
    
    if row['signal'] == 0:
        continue
    
    risk_amount = capital * RISK_PER_TRADE
    lot = min(MAX_LOT, risk_amount / (STOP_LOSS_PIPS * PIP_VALUE))
    
    entry = row['close']
    
    if row['signal'] == 1:  # BUY
        tp = entry + TAKE_PROFIT_PIPS * 0.0001
        sl = entry - STOP_LOSS_PIPS * 0.0001
        next_high = df.iloc[i+1]['high']
        next_low = df.iloc[i+1]['low']
        
        if next_low <= sl:
            pnl = -STOP_LOSS_PIPS * lot * PIP_VALUE
        elif next_high >= tp:
            pnl = TAKE_PROFIT_PIPS * lot * PIP_VALUE
        else:
            pnl = 0  # chưa hit TP/SL trong candle tiếp theo
            
    else:  # SELL
        tp = entry - TAKE_PROFIT_PIPS * 0.0001
        sl = entry + STOP_LOSS_PIPS * 0.0001
        next_high = df.iloc[i+1]['high']
        next_low = df.iloc[i+1]['low']
        
        if next_high >= sl:
            pnl = -STOP_LOSS_PIPS * lot * PIP_VALUE
        elif next_low <= tp:
            pnl = TAKE_PROFIT_PIPS * lot * PIP_VALUE
        else:
            pnl = 0
    
    capital += pnl
    trade_history.append({
        'datetime': row['datetime'],
        'signal': 'BUY' if row['signal']==1 else 'SELL',
        'entry': entry,
        'pnl': pnl,
        'capital': capital
    })

# ==========================
# 5️⃣ In kết quả
# ==========================
trade_df = pd.DataFrame(trade_history)
print(trade_df)
print(f"\nInitial Capital: {INITIAL_CAPITAL}")
print(f"Final Capital: {capital}")
print(f"Total Trades: {len(trade_history)}")
print(f"Net Profit: {capital - INITIAL_CAPITAL}")


KeyboardInterrupt: 

In [25]:
from pandas_ta.momentum.macd import macd
import pandas as pd
import numpy as np

# ==========================
# 1️⃣ Load dữ liệu + lọc ngày
# ==========================
DATA_PATH = '../data/EURUSD_5M.csv'
START_DATE = '2002-10-21'
END_DATE   = '2005-11-01'

df = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
df = df[(df['datetime'] >= START_DATE) & (df['datetime'] <= END_DATE)]

# ==========================
# 2️⃣ Tính MACD
# ==========================
macd_df = macd(df['close'])
df = pd.concat([df, macd_df], axis=1)

# ==========================
# 3️⃣ Tạo tín hiệu BUY/SELL
# ==========================
df['signal'] = 0
df.loc[df['MACD_12_26_9'] > df['MACDs_12_26_9'], 'signal'] = 1
df.loc[df['MACD_12_26_9'] < df['MACDs_12_26_9'], 'signal'] = -1

# ==========================
# 4️⃣ Parameters
# ==========================
INITIAL_CAPITAL = 1000
RISK_PER_TRADE = 0.01
MAX_LOT = 0.1
STOP_LOSS_PIPS = 20
TAKE_PROFIT_PIPS = 40
MAX_CANDLES_IN_POSITION = 60
PIP_VALUE = 10

# ==========================
# 5️⃣ Vectorized Backtest
# ==========================
# ==========================
capital = INITIAL_CAPITAL
trade_history = []

signals_idx = df.index[df['signal'] != 0].to_list()

for idx in signals_idx:
    if capital <= 0:
        break  # tránh vốn âm

    entry = df['close'].iloc[idx]
    signal = df['signal'].iloc[idx]

    lot = MAX_LOT  # fix lot, không nhân với capital nữa
    pnl = 0

    end_idx = min(idx + MAX_CANDLES_IN_POSITION, len(df)-1)
    candle_high = df['high'].iloc[idx+1:end_idx+1].values
    candle_low  = df['low'].iloc[idx+1:end_idx+1].values

    if signal == 1:  # BUY
        tp_price = entry + TAKE_PROFIT_PIPS*0.0001
        sl_price = entry - STOP_LOSS_PIPS*0.0001
        if np.any(candle_low <= sl_price):
            pnl = -STOP_LOSS_PIPS * lot * PIP_VALUE
        elif np.any(candle_high >= tp_price):
            pnl = TAKE_PROFIT_PIPS * lot * PIP_VALUE
    else:  # SELL
        tp_price = entry - TAKE_PROFIT_PIPS*0.0001
        sl_price = entry + STOP_LOSS_PIPS*0.0001
        if np.any(candle_high >= sl_price):
            pnl = -STOP_LOSS_PIPS * lot * PIP_VALUE
        elif np.any(candle_low <= tp_price):
            pnl = TAKE_PROFIT_PIPS * lot * PIP_VALUE

    capital += pnl
    trade_history.append({
        'datetime': df['datetime'].iloc[idx],
        'signal': 'BUY' if signal==1 else 'SELL',
        'entry': entry,
        'pnl': pnl,
        'capital': capital
    })


trade_df = pd.DataFrame(trade_history)
print(trade_df)
print(f"Final Capital: {capital}")


               datetime signal    entry   pnl  capital
0   2002-10-21 03:50:00    BUY  0.97545 -20.0    980.0
1   2002-10-21 03:55:00    BUY  0.97540 -20.0    960.0
2   2002-10-21 04:00:00    BUY  0.97530   0.0    960.0
3   2002-10-21 04:05:00    BUY  0.97480   0.0    960.0
4   2002-10-21 04:10:00    BUY  0.97500   0.0    960.0
..                  ...    ...      ...   ...      ...
321 2002-10-22 07:20:00   SELL  0.97680 -20.0     80.0
322 2002-10-22 07:25:00   SELL  0.97660 -20.0     60.0
323 2002-10-22 07:30:00   SELL  0.97665 -20.0     40.0
324 2002-10-22 07:35:00   SELL  0.97690 -20.0     20.0
325 2002-10-22 07:40:00   SELL  0.97715 -20.0      0.0

[326 rows x 5 columns]
Final Capital: 0.0
